# 04 Baseline vs LLM
Test your riskiest assumption on 25 cases with an LLM and with a rule that uses no model. Save one case level results file per method, a comparison table, and a scored choice of model, vendor and harness.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-04"))
OUTPUT.mkdir(parents=True, exist_ok=True)
STAGE1 = Path(os.environ.get("KIT_STAGE1_DIR", ROOT / "stage-01"))  # your Stage 1 probe_summary.csv
print("Kit folder found:", ROOT.name)

Step 2: write the assumption, the pass rule and the bar before you run

Write your own riskiest assumption in one sentence. The pass rule says when one case passes; the bar says when the assumption holds.

In [ ]:
ASSUMPTION = "Given a user's question, a model can pick the help page that answers it."
PASS_RULE = "A case passes when the picked page id is one of the case's expected page ids."
PASS_BAR = 0.80  # decided now, before any run: below this, the assumption fails
print(ASSUMPTION)
print("Pass rule:", PASS_RULE)
print("Pass bar:", PASS_BAR)

Step 3: load the cases and the help pages

In [ ]:
import json, re, time
import pandas as pd
from kit.llm import call_llm
from kit.prices import cost

cases = [json.loads(line) for line in open(CORPUS / "questions.jsonl")]
pages = []
for path in sorted(CORPUS.glob("*.md")):
    if path.name.lower() == "readme.md":
        continue
    _, head, body = path.read_text().split("---", 2)
    meta = {k.strip(): v.strip() for k, v in (line.split(":", 1) for line in head.strip().splitlines())}
    pages.append({"id": meta["id"], "title": meta["title"], "body": " ".join(body.split())})
print(len(cases), "cases,", len(pages), "pages")

Step 4: the baseline: a keyword rule over corpus/*.md

In [ ]:
STOP = {"the", "a", "an", "to", "i", "how", "do", "my", "is", "of", "and", "or", "in", "on", "for", "it", "can", "what"}

def words(text):
    return set(re.findall(r"[a-z0-9]+", text.lower())) - STOP

def keyword_rule(question):
    # Score each page: question words in the title count double, words in the body count once.
    q = words(question)
    return max(pages, key=lambda p: len(q & words(p["title"])) * 2 + len(q & words(p["body"])))["id"]

print(keyword_rule("How do I export a note as PDF?"))

Step 5: the LLM picks the page

In [ ]:
SYSTEM = ("Pick the help page that best answers the user's question. "
          'Reply with JSON only, like {"page_id": "qn-02"}, or {"page_id": "none"} if no page fits.')
PAGE_LIST = "\n".join(f"{p['id']}: {p['title']} | {p['body'][:300]}" for p in pages)

def llm_pick(question):
    r = call_llm(f"Pages:\n{PAGE_LIST}\n\nQuestion: {question}", system=SYSTEM, max_output_tokens=800)
    found = re.search(r"qn-\d+", r["text"])
    return (found.group(0) if found else "none"), r

pick, r = llm_pick("How do I export a note as PDF?")
print(pick, "|", r["model"], "|", r["input_tokens"], "input tokens")

Step 6: run both on every case and save one results file per method

Both files have the same case ids, so the comparison is on the exact same cases.

In [ ]:
llm_rows, rule_rows = [], []
for c in cases:
    start = time.perf_counter()
    base = keyword_rule(c["question"])
    base_latency = time.perf_counter() - start
    pick, r = llm_pick(c["question"])
    common = {"case_id": c["qid"], "input": c["question"], "expected": " ".join(c["page_ids"]), "pass_rule": PASS_RULE}
    rule_rows.append({**common, "method": "keyword rule", "system_output": base, "passed": base in c["page_ids"],
                      "latency_s": round(base_latency, 6), "cost_usd": 0.0})
    llm_rows.append({**common, "method": "LLM " + r["model"], "system_output": pick, "passed": pick in c["page_ids"],
                     "latency_s": r["latency_s"], "cost_usd": cost(r["input_tokens"], r["output_tokens"], r["model"])})
llm_results, baseline_results = pd.DataFrame(llm_rows), pd.DataFrame(rule_rows)
assert len(llm_results) >= 20 and list(llm_results.case_id) == list(baseline_results.case_id)
llm_results.to_csv(OUTPUT / "llm_results.csv", index=False)
baseline_results.to_csv(OUTPUT / "baseline_results.csv", index=False)
print(len(llm_results), "cases run, saved to", OUTPUT.name + "/llm_results.csv and", OUTPUT.name + "/baseline_results.csv")
llm_results.head()

Step 7: the comparison table

Write which method you would use now, and why, in 3 sentences or less. It goes into `comparison.md` with links to both results files.

In [ ]:
RECOMMENDATION = "Write here which method you would use now and why, in 3 sentences or less."

comparison = pd.DataFrame([{"method": df.method.iloc[0], "cases": len(df), "pass_rate": round(df.passed.mean(), 3),
                            "avg_latency_s": round(df.latency_s.mean(), 4), "avg_cost_usd": round(df.cost_usd.mean(), 6)}
                           for df in (baseline_results, llm_results)])
comparison.to_csv(OUTPUT / "comparison.csv", index=False)
table = ["| " + " | ".join(comparison.columns) + " |", "|" + " --- |" * len(comparison.columns)]
table += ["| " + " | ".join(str(v) for v in row) + " |" for row in comparison.itertuples(index=False)]
(OUTPUT / "comparison.md").write_text("\n".join([
    "# Baseline vs LLM on the same cases", "",
    f"Assumption: {ASSUMPTION}", f"Pass rule: {PASS_RULE} Bar: {PASS_BAR:.0%}.", "",
    *table, "",
    "Case level results: [baseline_results.csv](baseline_results.csv), [llm_results.csv](llm_results.csv).", "",
    f"**Which method I would use now:** {RECOMMENDATION}", ""]))
print("Saved", OUTPUT.name + "/comparison.md and comparison.csv")
comparison

Step 8: the verdict, and the cases where they disagree

In [ ]:
llm_rate = llm_results.passed.mean()
print("Assumption", "HOLDS" if llm_rate >= PASS_BAR else "FAILS", f"(LLM {llm_rate:.0%}, bar {PASS_BAR:.0%})")
print(f"LLM minus the free rule: {(llm_rate - baseline_results.passed.mean()) * 100:+.0f} points")
both = baseline_results[["case_id", "expected", "system_output", "passed"]].merge(
    llm_results[["case_id", "system_output", "passed"]], on="case_id", suffixes=("_rule", "_llm"))
both[both.passed_rule != both.passed_llm]

Step 9: score the next stack choice

Score at least two combinations of model, vendor and harness. Quality, cost and latency come from your own Stage 1 `probe_summary.csv`. The data terms score is yours: read each vendor's published data terms, write the date you read them and what they say, and score them 1 (bad for your users' data) to 5 (good). The kit's notes below are examples only: check the pages yourself and change them.

In [ ]:
from kit.llm import offline
MODEL_A = os.environ.get("PROBE_MODEL_A", "gemini-3.5-flash-lite")
MODEL_B = os.environ.get("PROBE_MODEL_B", "gemini-3.6-flash")
if offline():
    MODEL_A, MODEL_B = "mock-model-a", "mock-model-b"

CANDIDATES = [
    {"name": f"{MODEL_A} + Google Gemini API free tier + kit harness (app/app.py, keyword search, top 3)",
     "stage1_model": MODEL_A, "data_terms_score": 2,
     "terms_url": "https://ai.google.dev/gemini-api/terms", "terms_checked_on": "",
     "terms_note": "Example: on the free tier Google may use prompts and answers to improve its products."},
    {"name": f"{MODEL_B} + Google Gemini API paid tier + kit harness (app/app.py, keyword search, top 3)",
     "stage1_model": MODEL_B, "data_terms_score": 4,
     "terms_url": "https://ai.google.dev/gemini-api/terms", "terms_checked_on": "",
     "terms_note": "Example: on the paid tier Google says prompts are not used to improve its products."},
]
WEIGHTS = {"quality": 0.4, "cost": 0.2, "latency": 0.2, "data_terms": 0.2}  # must add up to 1

found = [path for path in [STAGE1 / "probe_summary.csv", Path("probe_summary.csv")] if path.exists()]
stage1 = pd.read_csv(found[0], index_col="model") if found else pd.DataFrame()
if not found:
    print("No Stage 1 probe_summary.csv found: run notebooks/01_probe_set.ipynb and put it in stage-01/. Scores use data terms only.")
rows = []
for cand in CANDIDATES:
    s = stage1.loc[cand["stage1_model"]] if cand["stage1_model"] in stage1.index else None
    rows.append({"candidate": cand["name"], "stage1_pass_rate": None if s is None else s.pass_rate,
                 "stage1_cost_per_task": None if s is None else s.cost_per_task,
                 "stage1_median_latency_s": None if s is None else s.median_latency_s,
                 "data_terms_score": cand["data_terms_score"]})
scores = pd.DataFrame(rows)
q = scores.stage1_pass_rate.fillna(0) * 5
cheap = 5 * scores.stage1_cost_per_task.min() / scores.stage1_cost_per_task
fast = 5 * scores.stage1_median_latency_s.min() / scores.stage1_median_latency_s
scores["score_out_of_5"] = (WEIGHTS["quality"] * q + WEIGHTS["cost"] * cheap.fillna(0) + WEIGHTS["latency"] * fast.fillna(0)
                            + WEIGHTS["data_terms"] * scores.data_terms_score).round(2)
scores = scores.sort_values("score_out_of_5", ascending=False)

lines = ["# Scored choice: model + vendor + harness", "",
         f"Weights: {WEIGHTS}. Quality = Stage 1 pass rate x 5. Cost and latency = 5 x best / this one. Data terms = my 1 to 5 score.",
         "Stage 1 numbers from: " + (f"[stage-01/probe_summary.csv](../stage-01/probe_summary.csv) and [stage-01/probe_runs.csv](../stage-01/probe_runs.csv)" if found else "no Stage 1 file found"), "",
         "| Candidate | Stage 1 pass rate | Cost per task (USD) | Median latency (s) | Data terms (1 to 5) | Score out of 5 |",
         "| --- | --- | --- | --- | --- | --- |"]
fmt = lambda v, f: "no Stage 1 number" if pd.isna(v) else format(v, f)
lines += [f"| {r.candidate} | {fmt(r.stage1_pass_rate, '.2f')} | {fmt(r.stage1_cost_per_task, '.6f')} | {fmt(r.stage1_median_latency_s, '.3f')} | {r.data_terms_score} | {r.score_out_of_5} |"
          for r in scores.itertuples()]
lines += ["", "## Data terms notes", ""]
for cand in CANDIDATES:
    lines.append(f"- {cand['name']}: read on {cand['terms_checked_on'] or '<date you read the page>'}, {cand['terms_url']}. {cand['terms_note']}")
lines += ["", f"**Chosen:** {scores.candidate.iloc[0]}"]
(OUTPUT / "stack_choice.md").write_text("\n".join(lines) + "\n")
if not all(cand["terms_checked_on"] for cand in CANDIDATES):
    print("Fill in terms_checked_on for every candidate after you read its data terms page.")
print("Saved", OUTPUT.name + "/stack_choice.md")
scores